In [1]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)


In [2]:
base_dir = Path("..", "data", "raw")
circularity_scenarios = ["base", "slow"]
climate_policy_scenario_dir = base_dir / 'SSP2'
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

circular_economy_scenario_dirs = {
    scenario: scenario_base_path / scenario
    for scenario in circularity_scenarios
}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
    circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)

def get_vema_prep():
    prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc

In [3]:
def get_buildings_prep():
    prep_data = bld.preprocess(base_dir, climate_policy_config, circular_economy_config)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [4]:
sectors = [get_vema_prep(), get_buildings_prep()]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

C:\Users\5982758\repos\image-materials\imagematerials\vehicles\preprocessing.py:334: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\repos\image-materials\imagematerials\vehicles\preprocessing.py:334: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\repos\image-materials\imagematerials\vehicles\preprocessing.py:334: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\repos\image-materials\imagematerials\vehicles\preprocessing.py:334: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\repos\image-materials\imagematerials\vehicles\preprocessing.py:334: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', f

implemented 'slow' for Vehicles


c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\indexes.py:469: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  index = pd.Index(np.asarray(array), **kwargs)
c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\indexes.py:469: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  index = pd.Index(np.asarray(array), **kwargs)
c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\indexes.py:469: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  index = pd.Index(np.asarray(array), **kwargs)
c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\indexes.py:469: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  index = pd.Index(np.asarray(array), **kwargs)
c:\Users\5982758\AppData\Local\anaconda3\envs\materi

implemented 'base' for Buildings
implemented 'slow' for Buildings


In [5]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [6]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [7]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [8]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [9]:
model.combi["summed_inflow_materials"][2020]

<xarray.DataArray (Region: 26, material: 18)> Size: 4kB
<Quantity([[3.27688925e+00 1.36348894e+07 0.00000000e+00 0.00000000e+00
  5.52834451e+04 0.00000000e+00 1.26374351e+08 1.96375976e+09
  0.00000000e+00 8.86465540e+01 0.00000000e+00 1.42076734e+05
  2.10935329e+04 1.49737285e+08 2.02994289e+08 0.00000000e+00
  1.30922932e+07 0.00000000e+00]
 [7.80054972e+01 3.35219239e+08 0.00000000e+00 0.00000000e+00
  1.73745885e+05 0.00000000e+00 3.19192852e+09 5.79780814e+10
  0.00000000e+00 3.19370076e+02 0.00000000e+00 7.16364553e+06
  9.60147070e+04 4.33461551e+09 5.99420719e+09 0.00000000e+00
  4.14604060e+08 0.00000000e+00]
 [0.00000000e+00 6.07695843e+07 0.00000000e+00 0.00000000e+00
  7.09560173e+04 0.00000000e+00 5.02569582e+08 3.75285015e+09
  0.00000000e+00 1.16273236e+02 0.00000000e+00 9.25643333e+04
  2.76164271e+04 2.64967308e+08 5.24457917e+08 0.00000000e+00
  2.42232673e+07 0.00000000e+00]
 [0.00000000e+00 6.40646176e+07 0.00000000e+00 0.00000000e+00
  3.44945051e+04 0.00000000e+00 5.06532368e+08 9.29185517e+09
  0.00000000e+00 5.56876757e+01 0.00000000e+00 1.87623296e+05
  1.35723791e+04 7.15928620e+08 1.19610954e+09 0.00000000e+00
  6.36368027e+07 0.00000000e+00]
...
 [4.54927772e+00 3.70192623e+08 1.09127400e+07 0.00000000e+00
  7.12579233e+04 5.11442898e+07 2.10350826e+09 1.00358406e+10
  2.70380373e+05 1.73377484e+02 0.00000000e+00 4.38709149e+07
  3.91269865e+04 6.03407557e+08 1.85917944e+09 0.00000000e+00
  5.72320060e+08 0.00000000e+00]
 [3.36017057e+01 2.68242378e+07 0.00000000e+00 0.00000000e+00
  1.64024188e+04 0.00000000e+00 2.05967005e+08 2.24254346e+09
  0.00000000e+00 2.81826041e+01 0.00000000e+00 8.51138168e+05
  7.83213842e+03 1.58278340e+08 3.24018582e+08 0.00000000e+00
  1.36523908e+07 0.00000000e+00]
 [4.52321501e+00 2.10657308e+08 0.00000000e+00 0.00000000e+00
  1.33623180e+05 0.00000000e+00 1.84065379e+09 9.31104006e+09
  0.00000000e+00 1.61353270e+02 0.00000000e+00 7.44841650e+05
  4.43108885e+04 6.12050710e+08 1.31986813e+09 0.00000000e+00
  5.75686505e+07 0.00000000e+00]
 [2.93112525e+00 1.07188171e+08 1.30740003e+06 0.00000000e+00
  6.99424572e+04 6.12733791e+06 7.79693490e+08 3.72389495e+09
  3.23929009e+04 9.45236599e+01 0.00000000e+00 5.41700042e+06
  2.44980510e+04 2.34231748e+08 5.91491790e+08 0.00000000e+00
  8.36110182e+07 0.00000000e+00]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * material  (material) <U9 648B 'Cement' 'Glass' 'Nd' 'Ti' ... 'Li' 'Cu' 'Ni'